# Inside the World of Sports Cars: Performance, Power & Price

**Exploratory Data Analysis (EDA)**  
Dataset: `sport_car_price.csv`

This notebook studies sports-car specifications and prices. It follows the assignment order: understand the data, define questions, clean the data, analyze individual variables and relationships, answer the questions, and conclude with actionable findings.

> Important data note: the source CSV contains 1,007 rows and 8 columns. Some numeric-looking fields contain labels such as `Electric`, `1,000+`, or `< 1.9`. These are handled transparently below rather than silently discarded.


## Step 1 — Understand the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

DATA_PATH = Path("sport_car_price.csv")
df_raw = pd.read_csv(DATA_PATH)

print("Source: the supplied Kaggle sports-car price CSV (local project copy).")
print("One row represents one listed sports-car model/specification record.")
print("Original shape:", df_raw.shape)
display(df_raw.head())
display(df_raw.describe(include="all").T)
print("\\nColumn names:", df_raw.columns.tolist())
print("\\nData types:")
display(df_raw.dtypes.to_frame("dtype"))


In [ ]:
column_guide = pd.DataFrame({
    "Column": df_raw.columns,
    "Meaning": [
        "Manufacturer or car brand",
        "Model name",
        "Model year",
        "Engine displacement or engine description",
        "Engine power in horsepower",
        "Engine torque in pound-feet",
        "Time to accelerate from 0 to 60 mph, in seconds",
        "Listed vehicle price in US dollars"
    ],
    "Type in source": [str(df_raw[c].dtype) for c in df_raw.columns]
})
display(column_guide)


## Step 2 — Questions to investigate

These questions are answerable using the available columns:

1. Which car manufacturers appear most often?
2. What is the distribution of sports-car prices?
3. What is the typical horsepower of cars in this dataset?
4. How does price relate to horsepower?
5. How does price relate to 0–60 mph acceleration time?
6. Which manufacturers have the highest median prices?
7. Do newer model years tend to have higher prices?
8. How does horsepower vary across manufacturers?

The questions include single-column analysis, relationships between two numeric columns, and comparisons between groups.


## Step 3 — Clean the data

In [ ]:
# Work from the original data so the number of exact repeated rows is measured consistently.
df = df_raw.copy()
missing_before = df.isna().sum()

print("Missing values before cleaning:")
display(missing_before.to_frame("missing_count"))
ax = missing_before[missing_before > 0].sort_values(ascending=False).plot(
    kind="bar", figsize=(9, 4), title="Missing values before cleaning"
)
ax.set_xlabel("Column")
ax.set_ylabel("Missing cells")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

duplicate_count = int(df.duplicated().sum())
print(f"Exact repeated rows: {duplicate_count}")
df = df.drop_duplicates().copy()
print("Shape after removing exact repeated rows:", df.shape)


In [ ]:
# Preserve the original engine description and derive a numeric engine-size feature.
engine_text = df["Engine Size (L)"].astype("string").str.strip()
df["Engine Type"] = np.select(
    [engine_text.str.contains("electric", case=False, na=False),
     engine_text.str.contains("hybrid", case=False, na=False)],
    ["Electric / electric-assisted", "Hybrid"],
    default="Combustion / unspecified"
)

# Parse numeric portions. For labels such as "Electric", there is no displacement value.
df["Engine Size (L)"] = pd.to_numeric(
    engine_text.str.extract(r"(\\d+(?:\\.\\d+)?)", expand=False),
    errors="coerce"
)

# Parse comma-separated numeric text and plus-suffixed values (e.g. "1,000+").
for col in ["Horsepower", "Torque (lb-ft)", "Price (in USD)"]:
    cleaned = df[col].astype("string").str.replace(",", "", regex=False)
    cleaned = cleaned.str.extract(r"(\\d+(?:\\.\\d+)?)", expand=False)
    df[col] = pd.to_numeric(cleaned, errors="coerce")

# "< 1.9" is represented as 1.9, the stated boundary, and interpreted as an approximate value.
accel_text = df["0-60 MPH Time (seconds)"].astype("string").str.replace(",", "", regex=False)
df["0-60 MPH Time (seconds)"] = pd.to_numeric(
    accel_text.str.extract(r"(\\d+(?:\\.\\d+)?)", expand=False),
    errors="coerce"
)
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

numeric_cols = [
    "Year", "Engine Size (L)", "Horsepower", "Torque (lb-ft)",
    "0-60 MPH Time (seconds)", "Price (in USD)"
]
print("Missing values after parsing text-formatted numbers:")
display(df[numeric_cols].isna().sum().to_frame("missing_count"))


In [ ]:
# Median-impute numeric gaps. This retains rows; Engine Type helps flag electric/hybrid records
# for which displacement may not be directly comparable.
imputation_log = {}
for col in numeric_cols:
    if df[col].isna().any():
        median_value = df[col].median()
        imputation_log[col] = float(median_value)
        df[col] = df[col].fillna(median_value)

print("Median values used for filling:", imputation_log)
print("Remaining missing values:")
display(df.isna().sum().to_frame("missing_count"))
print("Exact duplicates remaining:", int(df.duplicated().sum()))


In [ ]:
# IQR outlier checks on two important numeric columns.
def iqr_report(data, column):
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = int(((data[column] < lower) | (data[column] > upper)).sum())
    return {"column": column, "Q1": q1, "Q3": q3, "IQR": iqr,
            "lower_bound": lower, "upper_bound": upper, "outlier_count": count}

outlier_results = pd.DataFrame([
    iqr_report(df, "Price (in USD)"),
    iqr_report(df, "Horsepower")
])
display(outlier_results)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(y=df["Price (in USD)"], ax=axes[0])
axes[0].set_title("Price outliers (IQR check)")
axes[0].set_ylabel("Price (USD)")
sns.boxplot(y=df["Horsepower"], ax=axes[1])
axes[1].set_title("Horsepower outliers (IQR check)")
axes[1].set_ylabel("Horsepower")
plt.tight_layout()
plt.show()

print("Decision: retain the flagged values for now. Very expensive and powerful cars can be genuine specialist models.")
print("The IQR rule identifies unusual values; it does not prove they are errors. Extreme records should be checked against the source before removal.")


## Step 4 — Univariate analysis

In [ ]:
# Numeric distributions
numeric_to_plot = ["Price (in USD)", "Horsepower", "Torque (lb-ft)",
                   "0-60 MPH Time (seconds)", "Engine Size (L)", "Year"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for col, ax in zip(numeric_to_plot, axes.flat):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Number of cars")
plt.tight_layout()
plt.show()


In [ ]:
# Categorical distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df["Car Make"].value_counts().head(12).sort_values().plot(kind="barh", ax=axes[0])
axes[0].set_title("12 most frequent manufacturers")
axes[0].set_xlabel("Number of records")
axes[0].set_ylabel("Car make")

df["Engine Type"].value_counts().plot(kind="bar", ax=axes[1])
axes[1].set_title("Records by engine type")
axes[1].set_xlabel("Engine type")
axes[1].set_ylabel("Number of records")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()


## Step 5 — Bivariate analysis

In [ ]:
# Scatter plot: horsepower versus price
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="Horsepower", y="Price (in USD)", hue="Engine Type", alpha=0.75)
plt.title("Sports-car price versus horsepower")
plt.xlabel("Horsepower")
plt.ylabel("Price (USD)")
plt.tight_layout()
plt.show()

# Correlation heatmap
corr = df[numeric_cols].corr(numeric_only=True)
plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation between numeric specifications")
plt.tight_layout()
plt.show()


In [ ]:
# Numeric feature compared across categories
top_makes = df["Car Make"].value_counts().head(8).index
plot_df = df[df["Car Make"].isin(top_makes)]
plt.figure(figsize=(12, 5))
sns.boxplot(data=plot_df, x="Car Make", y="Horsepower")
plt.title("Horsepower distribution across the 8 most frequent manufacturers")
plt.xlabel("Manufacturer")
plt.ylabel("Horsepower")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


## Step 6 — Answer the 8 questions

In [ ]:
# Q1. Which manufacturers appear most often?
q1 = df["Car Make"].value_counts().head(10)
q1.sort_values().plot(kind="barh", figsize=(8, 4), title="Q1. Most frequent car manufacturers")
plt.xlabel("Number of records"); plt.ylabel("Car make"); plt.tight_layout(); plt.show()
print(f"Answer: {q1.index[0]} appears most often ({q1.iloc[0]} records).")

# Q2. What is the distribution of prices?
plt.figure(figsize=(8, 4))
sns.histplot(df["Price (in USD)"], bins=30, kde=True)
plt.title("Q2. Distribution of sports-car prices")
plt.xlabel("Price (USD)"); plt.ylabel("Number of cars"); plt.tight_layout(); plt.show()
print(f"Answer: median price is ${df['Price (in USD)'].median():,.0f}; the mean is ${df['Price (in USD)'].mean():,.0f}.")

# Q3. What is the typical horsepower?
plt.figure(figsize=(8, 4))
sns.histplot(df["Horsepower"], bins=30, kde=True)
plt.title("Q3. Horsepower distribution")
plt.xlabel("Horsepower"); plt.ylabel("Number of cars"); plt.tight_layout(); plt.show()
print(f"Answer: median horsepower is {df['Horsepower'].median():,.0f} HP.")

# Q4. How does price relate to horsepower?
plt.figure(figsize=(8, 4))
sns.regplot(data=df, x="Horsepower", y="Price (in USD)", scatter_kws={"alpha":0.45})
plt.title("Q4. Price versus horsepower")
plt.xlabel("Horsepower"); plt.ylabel("Price (USD)"); plt.tight_layout(); plt.show()
print(f"Answer: Pearson correlation = {df['Horsepower'].corr(df['Price (in USD)']):.2f}.")

# Q5. How does price relate to 0–60 time?
plt.figure(figsize=(8, 4))
sns.scatterplot(data=df, x="0-60 MPH Time (seconds)", y="Price (in USD)", alpha=0.6)
plt.title("Q5. Price versus 0–60 mph time")
plt.xlabel("0–60 mph time (seconds)"); plt.ylabel("Price (USD)"); plt.tight_layout(); plt.show()
print(f"Answer: correlation = {df['0-60 MPH Time (seconds)'].corr(df['Price (in USD)']):.2f}.")

# Q6. Which makes have the highest median prices?
q6 = df.groupby("Car Make")["Price (in USD)"].median().sort_values(ascending=False).head(10)
q6.sort_values().plot(kind="barh", figsize=(8, 5), title="Q6. Manufacturers with highest median prices")
plt.xlabel("Median price (USD)"); plt.ylabel("Car make"); plt.tight_layout(); plt.show()
print("Answer: highest median-price manufacturer in the displayed group:", q6.index[0])

# Q7. Do newer model years tend to have higher prices?
year_price = df.groupby("Year", as_index=False)["Price (in USD)"].median()
sns.lineplot(data=year_price, x="Year", y="Price (in USD)", marker="o")
plt.title("Q7. Median price by model year")
plt.xlabel("Model year"); plt.ylabel("Median price (USD)"); plt.tight_layout(); plt.show()
print("Answer: compare the plotted yearly medians; the relationship is descriptive, not proof that age causes price.")

# Q8. How does horsepower vary by manufacturer?
q8_makes = df["Car Make"].value_counts().head(10).index
q8 = df[df["Car Make"].isin(q8_makes)]
plt.figure(figsize=(12, 5))
sns.barplot(data=q8, x="Car Make", y="Horsepower", estimator="median", errorbar=None)
plt.title("Q8. Median horsepower by frequent manufacturer")
plt.xlabel("Car make"); plt.ylabel("Median horsepower"); plt.xticks(rotation=35, ha="right")
plt.tight_layout(); plt.show()
print("Answer: the bars compare median horsepower among the 10 most frequent manufacturers.")


## Step 7 — Conclusion

In [ ]:
print("FINAL DATASET SHAPE:", df.shape)
print("\\nThree data-backed summary findings:")
print(f"1. The median listed price is ${df['Price (in USD)'].median():,.0f}.")
print(f"2. The median horsepower is {df['Horsepower'].median():,.0f} HP.")
print(f"3. Price and horsepower have a Pearson correlation of {df['Horsepower'].corr(df['Price (in USD)']):.2f}.")

print("\\nActionable suggestion:")
print("A buyer comparing performance-focused cars should compare price, horsepower, and 0–60 time together rather than relying on horsepower alone.")

# Save the cleaned data for the Streamlit app (same folder as this notebook).
df.to_csv("sport_car_price_cleaned.csv", index=False)
print("\\nSaved: sport_car_price_cleaned.csv")
